In [6]:
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path

outputs_dir = Path.cwd().parent / "outputs"
outputs_dir.mkdir(exist_ok=True)

summary_data = [
    {
        "Тип ошибки": "Лексический сдвиг категории",
        "Количество случаев": 4,
        "Процент от всех ошибок": "44.4%",
        "Затронутые модели": "mpnet, MiniLM",
        "Затронутые вопросы": "q_01, q_24",
        "Механизм возникновения": "Глагол «проверить» / слово «verification» перетягивает выдачу в категорию validation или auth, даже когда запрос про другую техническую операцию.",
        "Причина": "В векторном пространстве кластеры с глаголами validate, check, проверить очень плотные. Модель не смогла сохранить контекст (токен/хэш), лексический триггер перебил техническую специфику."
    },
    {
        "Тип ошибки": "Размытие технического контекста",
        "Количество случаев": 2,
        "Процент от всех ошибок": "22.2%",
        "Затронутые модели": "e5-small, MiniLM",
        "Затронутые вопросы": "q_08",
        "Механизм возникновения": "Модель реагирует на признак «большое количество» / «массовость» вместо целевой операции (вставка в БД).",
        "Причина": "В векторном пространстве признак «массовость/большое количество» близок к функциям, которые обрабатывают объёмы данных (chunk, count, format_bytes). Технический аспект «вставить список» утратил значимость на фоне количественного признака."
    },
    {
        "Тип ошибки": "Кросс-языковое совпадение",
        "Количество случаев": 3,
        "Процент от всех ошибок": "33.3%",
        "Затронутые модели": "MiniLM",
        "Затронутые вопросы": "q_08, q_11, q_24",
        "Механизм возникновения": "Модель находит семантически идентичную функцию, но на другом языке программирования (Java вместо Python). Строгая метрика по correct_chunk_id считает это ошибкой, хотя смысл понят верно.",
        "Причина": "Модели кодируют смысл, а не синтаксис. Описания функций на русском языке идентичны для Python и Java версий, поэтому в эмбеддинг-пространстве они оказываются рядом. Это не ошибка понимания, а артефакт строгой оценки по ID."
    }
]

df_summary = pd.DataFrame(summary_data)

# CSS стили для таблицы
html_summary_code = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Типология ошибок</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            margin: 30px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
        }}
        th {{
            background-color: #2c3e50;
            color: white;
            font-weight: bold;
            text-align: center;
            padding: 12px 8px;
            border: 1px solid #466;
        }}
        td {{
            border: 1px solid #ddd;
            padding: 10px 8px;
            vertical-align: top;
            text-align: left;
        }}
        td:first-child {{
            background-color: #f8f9fa;
            font-weight: bold;
        }}
        td:nth-child(2), td:nth-child(3), td:nth-child(4), td:nth-child(5) {{
            text-align: center;
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        tr:hover {{
            background-color: #f0f0f0;
        }}
    </style>
</head>
<body>

<h2 style="text-align: center; color: #2c3e50;">Типология ошибок семантического поиска</h2>

<table>
    <thead>
        <tr>
            <th style="width: 180px;">Тип ошибки</th>
            <th style="width: 60px;">Кол-во</th>
            <th style="width: 70px;">Процент</th>
            <th style="width: 150px;">Затронутые модели</th>
            <th style="width: 100px;">Вопросы</th>
            <th style="width: 220px;">Механизм возникновения</th>
            <th style="width: 350px;">Причина</th>
        </tr>
    </thead>
    <tbody>
"""

for _, row in df_summary.iterrows():
    html_summary_code += f"""
        <tr>
            <td>{row['Тип ошибки']}</td>
            <td style="text-align: center;"><strong>{row['Количество случаев']}</strong></td>
            <td style="text-align: center;">{row['Процент от всех ошибок']}</td>
            <td>{row['Затронутые модели']}</td>
            <td style="text-align: center;">{row['Затронутые вопросы']}</td>
            <td>{row['Механизм возникновения']}</td>
            <td>{row['Причина']}</td>
        </tr>
    """

html_summary_code += """
    </tbody>
</table>

</body>
</html>
"""

# Отображаем в Jupyter
display(HTML(html_summary_code))

# Сохраняем HTML
with open(outputs_dir / "summary_error_types.html", "w", encoding="utf-8") as f:
    f.write(html_summary_code)
print(f"\nСводная таблица сохранена в {outputs_dir / 'summary_error_types.html'}")

# Сохраняем в CSV
df_summary.to_csv(outputs_dir / "summary_error_types.csv", index=False, encoding="utf-8-sig")
print(f"CSV сохранён в {outputs_dir / 'summary_error_types.csv'}")

print("\n" + "="*60)
print("Файлы сохранены в папку outputs:")
print("  - summary_error_types.html")
print("  - summary_error_types.csv")
print("="*60)

Тип ошибки,Кол-во,Процент,Затронутые модели,Вопросы,Механизм возникновения,Причина
Лексический сдвиг категории,4,44.4%,"mpnet, MiniLM","q_01, q_24","Глагол «проверить» / слово «verification» перетягивает выдачу в категорию validation или auth, даже когда запрос про другую техническую операцию.","В векторном пространстве кластеры с глаголами validate, check, проверить очень плотные. Модель не смогла сохранить контекст (токен/хэш), лексический триггер перебил техническую специфику."
Размытие технического контекста,2,22.2%,"e5-small, MiniLM",q_08,Модель реагирует на признак «большое количество» / «массовость» вместо целевой операции (вставка в БД).,"В векторном пространстве признак «массовость/большое количество» близок к функциям, которые обрабатывают объёмы данных (chunk, count, format_bytes). Технический аспект «вставить список» утратил значимость на фоне количественного признака."
Кросс-языковое совпадение,3,33.3%,MiniLM,"q_08, q_11, q_24","Модель находит семантически идентичную функцию, но на другом языке программирования (Java вместо Python). Строгая метрика по correct_chunk_id считает это ошибкой, хотя смысл понят верно.","Модели кодируют смысл, а не синтаксис. Описания функций на русском языке идентичны для Python и Java версий, поэтому в эмбеддинг-пространстве они оказываются рядом. Это не ошибка понимания, а артефакт строгой оценки по ID."



Сводная таблица сохранена в C:\CPP\semantic-search-case\outputs\summary_error_types.html
CSV сохранён в C:\CPP\semantic-search-case\outputs\summary_error_types.csv

Файлы сохранены в папку outputs:
  - summary_error_types.html
  - summary_error_types.csv
